# Todorov Phase 1: Language Modeling Baseline

Eptesicus Laboratories

Trains Todorov hybrid architecture (KDA + Mamba-3 + MLA) and a Transformer baseline
on byte-level WikiText-2. Evaluates Phase 1 gates: BPB ratio, spike health.

In [ ]:
# Configuration - edit these before running
MAX_STEPS = 500        # 500 for dev iteration, 2000 for full run
EVAL_INTERVAL = 100    # evaluate every N steps
BATCH_SIZE = 32
LR = 3e-4
SEQ_LEN = 256
D_MODEL = 256
N_LAYERS = 8
USE_SPIKES = True
SPIKE_ALPHA = 1.0

In [ ]:
!pip install -q datasets matplotlib

In [ ]:
from __future__ import annotations

import gc
import json
import math
import os
import time
import zipfile
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from torch.utils.data import DataLoader, Dataset

def _select_device() -> torch.device:
    if not torch.cuda.is_available():
        return torch.device('cpu')
    try:
        cap = torch.cuda.get_device_capability()
        if cap[0] < 7:
            print(f'GPU sm_{cap[0]}{cap[1]} not supported, using CPU')
            return torch.device('cpu')
    except Exception:
        pass
    return torch.device('cuda')

DEVICE = _select_device()
SEED = 42
OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)

torch.manual_seed(SEED)
np.random.seed(SEED)
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')

In [ ]:
# === MODEL CODE (self-contained) ===

class RotaryPE(nn.Module):
    def __init__(self, dim, base=10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer('inv_freq', inv_freq)
        self._cache_len = 0
        self._cos = None
        self._sin = None

    def _build(self, length, device):
        if length <= self._cache_len and self._cos is not None:
            return
        self._cache_len = length
        t = torch.arange(length, device=device, dtype=self.inv_freq.dtype)
        freqs = torch.outer(t, self.inv_freq.to(device))
        emb = torch.cat([freqs, freqs], dim=-1)
        self._cos = emb.cos()
        self._sin = emb.sin()

    def forward(self, x, offset=0):
        self._build(x.shape[-2] + offset, x.device)
        sl = x.shape[-2]
        return self._cos[offset:offset+sl], self._sin[offset:offset+sl]


def rotary_apply(x, cos, sin):
    d = x.shape[-1]
    x1, x2 = x[..., :d//2], x[..., d//2:]
    c, s = cos[..., :d//2], sin[..., :d//2]
    return torch.cat([x1*c - x2*s, x2*c + x1*s], dim=-1)


class TernaryQuantizer(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, threshold, training=True):
        ctx.save_for_backward(x, threshold)
        out = torch.zeros_like(x)
        out[x > threshold] = 1.0
        out[x < -threshold] = -1.0
        return out

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.clone(), None, None


class AdaptiveSpike(nn.Module):
    def __init__(self, alpha=1.0):
        super().__init__()
        self.alpha = nn.Parameter(torch.tensor(alpha))
        self.register_buffer('running_density', torch.tensor(0.0))
        self.register_buffer('n_updates', torch.tensor(0))

    def forward(self, x):
        threshold = torch.clamp(self.alpha * x.abs().mean(), 0.01, 10.0)
        spikes = TernaryQuantizer.apply(x, threshold, self.training)
        if self.training:
            with torch.no_grad():
                d = (spikes != 0).float().mean()
                self.running_density = 0.99 * self.running_density + 0.01 * d
                self.n_updates += 1
        return spikes


class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d))
        self.eps = eps

    def forward(self, x):
        rms = (x.float().pow(2).mean(-1, keepdim=True) + self.eps).rsqrt()
        return (x.float() * rms).to(x.dtype) * self.weight


class KDALayer(nn.Module):
    def __init__(self, d, nh, hd, spikes=False, alpha=1.0):
        super().__init__()
        self.nh, self.hd = nh, hd
        inner = nh * hd
        self.q = nn.Linear(d, inner, bias=False)
        self.k = nn.Linear(d, inner, bias=False)
        self.v = nn.Linear(d, inner, bias=False)
        self.o = nn.Linear(inner, d, bias=False)
        self.beta_proj = nn.Linear(d, nh, bias=True)
        self.alpha_log = nn.Parameter(torch.zeros(nh, hd) - 2.0)
        self.rope = RotaryPE(hd)
        self.k_spike = AdaptiveSpike(alpha) if spikes else None
        self.v_spike = AdaptiveSpike(alpha) if spikes else None

    def forward(self, x, state=None, offset=0):
        B, T, _ = x.shape
        aux = {}
        qr = self.q(x).view(B, T, self.nh, self.hd)
        kr = self.k(x).view(B, T, self.nh, self.hd)
        vr = self.v(x).view(B, T, self.nh, self.hd)
        cos, sin = self.rope(qr, offset)
        c, s = cos.unsqueeze(1), sin.unsqueeze(1)
        qr = rotary_apply(qr.transpose(1,2), c, s).transpose(1,2)
        kr = rotary_apply(kr.transpose(1,2), c, s).transpose(1,2)
        if self.k_spike:
            aux['pre_k'] = kr.detach()
            kr = self.k_spike(kr)
            aux['k_spikes'] = kr.detach()
        if self.v_spike:
            aux['pre_v'] = vr.detach()
            vr = self.v_spike(vr)
            aux['v_spikes'] = vr.detach()
        alpha = torch.sigmoid(self.alpha_log).unsqueeze(0).unsqueeze(-1)
        beta = torch.sigmoid(self.beta_proj(x)).unsqueeze(-1)
        if state is None:
            state = torch.zeros(B, self.nh, self.hd, self.hd, device=x.device, dtype=x.dtype)
        outs = []
        for t in range(T):
            bt = beta[:, t].unsqueeze(-1)
            state = alpha * state + bt * torch.einsum('bhd,bhe->bhde', kr[:,t], vr[:,t])
            outs.append(torch.einsum('bhd,bhde->bhe', qr[:,t], state))
        out = torch.stack(outs, 1).reshape(B, T, -1)
        aux['state_norm'] = state.norm().item()
        return self.o(out), state, aux


class Mamba3Layer(nn.Module):
    def __init__(self, d, ds=16, expand=2):
        super().__init__()
        di = d * expand
        self.in_proj = nn.Linear(d, di*2, bias=False)
        self.A_log = nn.Parameter(torch.randn(di, ds).uniform_(-4, -1))
        self.B_proj = nn.Linear(d, ds, bias=False)
        self.C_proj = nn.Linear(d, ds, bias=False)
        self.dt_proj = nn.Linear(d, di, bias=True)
        self.dt_bias = nn.Parameter(torch.zeros(di) - 4.0)
        self.norm = nn.LayerNorm(di)
        self.out_proj = nn.Linear(di, d, bias=False)
        self.di, self.ds = di, ds

    def forward(self, x, state=None, offset=0):
        B, T, _ = x.shape
        xz = self.in_proj(x)
        xi, z = xz.chunk(2, dim=-1)
        z = F.silu(z)
        A = -torch.exp(self.A_log)
        Bm = self.B_proj(x)
        C = self.C_proj(x)
        dt = F.softplus(self.dt_proj(x) + self.dt_bias)
        dtA = dt.unsqueeze(-1) * A.unsqueeze(0).unsqueeze(0)
        A_bar = (1 + dtA/2) / (1 - dtA/2)
        B_bar = dt.unsqueeze(-1) / (1 - dtA/2)
        if state is None:
            state = torch.zeros(B, self.di, self.ds, device=x.device, dtype=x.dtype)
        outs = []
        for t in range(T):
            Bt = Bm[:,t].unsqueeze(1).expand(-1, self.di, -1)
            xt = xi[:,t].unsqueeze(-1)
            state = A_bar[:,t] * state + B_bar[:,t] * Bt * xt
            Ct = C[:,t].unsqueeze(1).expand(-1, self.di, -1)
            outs.append((state * Ct).sum(-1))
        out = torch.stack(outs, 1) * z
        return self.out_proj(self.norm(out)), state, {'state_util': (state.abs()>1e-6).float().mean().item()}


class MLALayer(nn.Module):
    def __init__(self, d, dc=64, dr=16, nh=4):
        super().__init__()
        self.dc, self.dr, self.nh = dc, dr, nh
        self.hd = d // nh
        self.kv_down = nn.Linear(d, dc, bias=False)
        self.k_up = nn.Linear(dc, d, bias=False)
        self.v_up = nn.Linear(dc, d, bias=False)
        self.q_proj = nn.Linear(d, d, bias=False)
        self.q_rope = nn.Linear(d, dr, bias=False)
        self.k_rope = nn.Linear(dc, dr, bias=False)
        self.o_proj = nn.Linear(d, d, bias=False)
        inv = 1.0 / (10000.0 ** (torch.arange(0, dr, 2).float() / dr))
        self.register_buffer('inv_freq', inv)

    def _rope(self, sl, off, dev):
        t = torch.arange(off, off+sl, device=dev, dtype=self.inv_freq.dtype)
        f = torch.outer(t, self.inv_freq)
        e = torch.cat([f, f], -1)
        return e.cos(), e.sin()

    def _rap(self, x, c, s):
        d = x.shape[-1]
        x1, x2 = x[..., :d//2], x[..., d//2:]
        return torch.cat([x1*c[...,:d//2]-x2*s[...,:d//2], x2*c[...,:d//2]+x1*s[...,:d//2]], -1)

    def forward(self, x, kv_cache=None, offset=0):
        B, T, _ = x.shape
        ckv = self.kv_down(x)
        kr_shared = self.k_rope(ckv)
        cos, sin = self._rope(T, offset, x.device)
        cos, sin = cos.unsqueeze(0), sin.unsqueeze(0)
        kr_shared = self._rap(kr_shared, cos, sin)
        entry = torch.cat([ckv, kr_shared], -1)
        if kv_cache is not None:
            entry = torch.cat([kv_cache, entry], 1)
        cached_ckv = entry[..., :self.dc]
        cached_kr = entry[..., self.dc:]
        tl = entry.shape[1]
        k = self.k_up(cached_ckv).view(B, tl, self.nh, self.hd).transpose(1,2)
        v = self.v_up(cached_ckv).view(B, tl, self.nh, self.hd).transpose(1,2)
        q = self.q_proj(x).view(B, T, self.nh, self.hd).transpose(1,2)
        qr_shared = self.q_rope(x)
        qc, qs = self._rope(T, offset, x.device)
        qc, qs = qc.unsqueeze(0), qs.unsqueeze(0)
        qr_shared = self._rap(qr_shared, qc, qs)
        ac = torch.matmul(q, k.transpose(-2,-1))
        qr_exp = qr_shared.unsqueeze(2).expand(-1,-1,self.nh,-1).transpose(1,2)
        kr_exp = cached_kr.unsqueeze(2).expand(-1,-1,self.nh,-1).transpose(1,2)
        ar = torch.matmul(qr_exp, kr_exp.transpose(-2,-1))
        scale = math.sqrt(self.hd + self.dr)
        scores = (ac + ar) / scale
        if T > 1:
            mask = torch.triu(torch.full((T, tl), float('-inf'), device=x.device), diagonal=tl-T+1)
            scores = scores + mask.unsqueeze(0).unsqueeze(0)
        out = torch.matmul(F.softmax(scores, -1), v).transpose(1,2).reshape(B, T, -1)
        return self.o_proj(out), entry, {'compression': (2*self.nh*self.hd)/(self.dc+self.dr)}


class TransformerLayer(nn.Module):
    def __init__(self, d, nh=4):
        super().__init__()
        self.nh = nh
        self.hd = d // nh
        self.qkv = nn.Linear(d, 3*d, bias=False)
        self.o_proj = nn.Linear(d, d, bias=False)
        self.rope = RotaryPE(d // nh)

    def forward(self, x, state=None, offset=0):
        B, T, _ = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.nh, self.hd)
        q, k, v = qkv[:,:,0], qkv[:,:,1], qkv[:,:,2]
        cos, sin = self.rope(q, offset)
        c, s = cos.unsqueeze(1), sin.unsqueeze(1)
        q = rotary_apply(q.transpose(1,2), c, s)
        k = rotary_apply(k.transpose(1,2), c, s)
        v = v.transpose(1,2)
        scores = torch.matmul(q, k.transpose(-2,-1)) / math.sqrt(self.hd)
        mask = torch.triu(torch.full((T, T), float('-inf'), device=x.device), diagonal=1)
        out = torch.matmul(F.softmax(scores + mask, -1), v).transpose(1,2).reshape(B, T, -1)
        return self.o_proj(out), None, {}


class SwiGLU(nn.Module):
    def __init__(self, d, ratio=2.25):
        super().__init__()
        h = int(d * ratio)
        h = ((h + 63) // 64) * 64
        self.gate = nn.Linear(d, h, bias=False)
        self.up = nn.Linear(d, h, bias=False)
        self.down = nn.Linear(h, d, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))


class Block(nn.Module):
    def __init__(self, lt, d, nh, hd, ds, exp, dc, dr, ratio, spikes, alpha):
        super().__init__()
        self.lt = lt
        self.n1 = RMSNorm(d)
        self.n2 = RMSNorm(d)
        self._sk = 'kv_cache' if lt == 'MLA' else 'state'
        if lt == 'KDA':
            self.attn = KDALayer(d, nh, hd, spikes, alpha)
        elif lt == 'Mamba3':
            self.attn = Mamba3Layer(d, ds, exp)
        elif lt == 'MLA':
            self.attn = MLALayer(d, dc, dr, nh)
        elif lt == 'Transformer':
            self.attn = TransformerLayer(d, nh)
            self._sk = 'state'
        self.mlp = SwiGLU(d, ratio)

    def forward(self, x, state=None, offset=0):
        r = x
        a, ns, aux = self.attn(self.n1(r), **{self._sk: state, 'offset': offset})
        x = r + a
        x = x + self.mlp(self.n2(x))
        return x, ns, aux


class LM(nn.Module):
    def __init__(self, d, n, vocab, nh, hd, ds, exp, dc, dr, ratio, pattern, spikes, alpha):
        super().__init__()
        self.n_layers = n
        self.emb = nn.Embedding(vocab, d)
        nn.init.normal_(self.emb.weight, std=0.02)
        lts = list(pattern) * (n // len(pattern))
        self.blocks = nn.ModuleList([
            Block(lts[i], d, nh, hd, ds, exp, dc, dr, ratio, spikes, alpha)
            for i in range(n)
        ])
        self.norm = RMSNorm(d)
        self.head = nn.Linear(d, vocab, bias=False)
        self.head.weight = self.emb.weight

    def forward(self, ids, states=None, offset=0):
        x = self.emb(ids)
        if states is None:
            states = [None] * self.n_layers
        ns, all_aux = [], {}
        for i, b in enumerate(self.blocks):
            x, s, aux = b(x, states[i], offset)
            ns.append(s)
            all_aux[i] = aux
        return self.head(self.norm(x)), ns, all_aux

    def count_params(self):
        return sum(p.numel() for p in self.parameters())

print('Model code loaded.')

In [ ]:
# === DATA ===

class ByteTextDataset(Dataset):
    def __init__(self, data, seq_len):
        self.data = torch.frombuffer(bytearray(data), dtype=torch.uint8).long()
        self.seq_len = seq_len

    def __len__(self):
        return max(1, (len(self.data) - 1) // self.seq_len)

    def __getitem__(self, idx):
        start = idx * self.seq_len
        end = min(start + self.seq_len + 1, len(self.data))
        return self.data[start:end]

def collate_fn(batch):
    max_len = max(b.shape[0] for b in batch)
    padded = torch.zeros(len(batch), max_len, dtype=torch.long)
    for i, b in enumerate(batch):
        padded[i, :b.shape[0]] = b
    return padded

from datasets import load_dataset
ds = load_dataset('wikitext', 'wikitext-2-raw-v1')
train_text = '\n'.join(ds['train']['text'])
val_text = '\n'.join(ds['validation']['text'])
train_data = train_text.encode('utf-8')
val_data = val_text.encode('utf-8')
print(f'Train: {len(train_data):,} bytes, Val: {len(val_data):,} bytes')

train_ds = ByteTextDataset(train_data, SEQ_LEN)
val_ds = ByteTextDataset(val_data, SEQ_LEN)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, drop_last=True)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

In [ ]:
# === TRAIN FUNCTION ===

@torch.no_grad()
def evaluate(model, dataloader, max_batches=50):
    model.eval()
    total_loss, total_tokens = 0.0, 0
    for i, batch in enumerate(dataloader):
        if i >= max_batches: break
        batch = batch.to(DEVICE)
        logits, _, _ = model(batch[:, :-1])
        loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), batch[:, 1:].reshape(-1), reduction='sum')
        total_loss += loss.item()
        total_tokens += batch[:, 1:].numel()
    model.train()
    mean_loss = total_loss / max(total_tokens, 1)
    return {'mean_loss': mean_loss, 'bpb': mean_loss / math.log(2)}


def train_model(model, name, train_dl, val_dl, max_steps, lr, eval_interval):
    print(f'\n=== Training {name} ({model.count_params():,} params) ===')
    model = model.to(DEVICE)

    decay_p, no_decay_p = [], []
    for pn, p in model.named_parameters():
        if not p.requires_grad: continue
        if p.ndim <= 1 or 'norm' in pn or 'bias' in pn:
            no_decay_p.append(p)
        else:
            decay_p.append(p)
    optimizer = torch.optim.AdamW([
        {'params': decay_p, 'weight_decay': 0.01},
        {'params': no_decay_p, 'weight_decay': 0.0},
    ], lr=lr, betas=(0.9, 0.95))

    def lr_fn(step):
        if step < 200: return step / 200
        progress = (step - 200) / max(1, max_steps - 200)
        return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(1.0, progress)))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_fn)

    history = {'train_loss': [], 'val_bpb': [], 'steps': [], 'spike_frs': []}
    best_bpb = float('inf')
    step = 0
    start = time.time()

    for epoch in range(100):
        if step >= max_steps: break
        for batch in train_dl:
            if step >= max_steps: break
            batch = batch.to(DEVICE)
            logits, _, aux = model(batch[:, :-1])
            loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), batch[:, 1:].reshape(-1))
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            if step % 50 == 0:
                frs = []
                for la in aux.values():
                    if isinstance(la, dict):
                        for k in ('k_spikes', 'v_spikes'):
                            if k in la:
                                frs.append((la[k] != 0).float().mean().item())
                fr_str = f' FR={np.mean(frs):.3f}' if frs else ''
                elapsed = time.time() - start
                print(f'  step {step:>5}/{max_steps} loss={loss.item():.4f} lr={scheduler.get_last_lr()[0]:.2e}{fr_str} [{elapsed:.0f}s]')
                history['train_loss'].append(loss.item())
                history['steps'].append(step)
                history['spike_frs'].append(np.mean(frs) if frs else 0)

            if step > 0 and step % eval_interval == 0:
                val = evaluate(model, val_dl)
                history['val_bpb'].append(val['bpb'])
                if val['bpb'] < best_bpb:
                    best_bpb = val['bpb']
                print(f'  ** VAL bpb={val["bpb"]:.4f} best={best_bpb:.4f}')

            step += 1

    final = evaluate(model, val_dl)
    history['val_bpb'].append(final['bpb'])
    if final['bpb'] < best_bpb:
        best_bpb = final['bpb']
    total_time = time.time() - start
    print(f'  DONE: final_bpb={final["bpb"]:.4f} best={best_bpb:.4f} time={total_time:.0f}s')
    return {'best_bpb': best_bpb, 'final_bpb': final['bpb'], 'history': history, 'time': total_time}

print('Training functions loaded.')

In [ ]:
# === TRAIN TODOROV ===

todorov = LM(
    d=D_MODEL, n=N_LAYERS, vocab=256, nh=4, hd=D_MODEL//4,
    ds=16, exp=2, dc=D_MODEL//4, dr=16, ratio=2.25,
    pattern=('KDA','KDA','KDA','Mamba3','KDA','KDA','KDA','MLA'),
    spikes=USE_SPIKES, alpha=SPIKE_ALPHA,
)
t_result = train_model(todorov, 'Todorov', train_dl, val_dl, MAX_STEPS, LR, EVAL_INTERVAL)

del todorov
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

In [ ]:
# === TRAIN TRANSFORMER BASELINE ===

baseline = LM(
    d=D_MODEL, n=N_LAYERS, vocab=256, nh=4, hd=D_MODEL//4,
    ds=16, exp=2, dc=D_MODEL//4, dr=16, ratio=2.25,
    pattern=('Transformer',)*8,
    spikes=False, alpha=1.0,
)
b_result = train_model(baseline, 'Transformer', train_dl, val_dl, MAX_STEPS, LR, EVAL_INTERVAL)

del baseline
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

In [ ]:
# === RESULTS ===

ratio = t_result['best_bpb'] / max(b_result['best_bpb'], 1e-6)
gate_a = ratio <= 1.5

print(f'Todorov BPB:     {t_result["best_bpb"]:.4f}')
print(f'Baseline BPB:    {b_result["best_bpb"]:.4f}')
print(f'BPB Ratio:       {ratio:.4f}')
print(f'Gate A (<=1.5x): {"PASS" if gate_a else "FAIL"}')
print()

frs = t_result['history']['spike_frs']
mean_fr = np.mean(frs) if frs else 0
print(f'Mean spike FR:   {mean_fr:.4f}')
print(f'Gate B (FR):     {"PASS" if 0.3 <= mean_fr <= 0.6 else "FAIL"}')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(t_result['history']['steps'], t_result['history']['train_loss'], label='Todorov')
axes[0].plot(b_result['history']['steps'], b_result['history']['train_loss'], label='Transformer')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(t_result['history']['val_bpb'], 'o-', label='Todorov')
axes[1].plot(b_result['history']['val_bpb'], 's-', label='Transformer')
axes[1].set_title('Validation BPB')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

if any(f > 0 for f in frs):
    axes[2].plot(t_result['history']['steps'][:len(frs)], frs)
    axes[2].axhline(y=0.3, color='r', ls='--', alpha=0.5)
    axes[2].axhline(y=0.6, color='r', ls='--', alpha=0.5)
axes[2].set_title('Spike Firing Rate')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=150)
plt.show()